In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================================
# TRAINING CONFIGURATION
# ============================================================================

EPISODES = 10000
GAMMA = 0.69
STEPS_PER_EPISODE = 3600
FORCE_RETRAIN = True
EPSILON = 0.1
# ============================================================================
# PK/PD PARAMETERS (Schnider Model)
# ============================================================================

V1, V2, V3 = 4.27, 18.9, 238.0
k10, k12, k21, k13, k31 = 0.38, 0.30, 0.20, 0.19, 0.0035
ke0 = 0.17
BIS_0, BIS_MAX, EC50, HILL = 95.0, 75.0, 3.5, 2.5
BIS_TARGET = 50.0

# ============================================================================
# RL CONFIGURATION
# ============================================================================

ACTIONS = [0.0, 0.5, 1.0, 2.0, 3.0, 4.0, 6.0]
BINS_PER_FEAT = 10
NUM_STATES = BINS_PER_FEAT**6  # 1,000,000 states (6 fuzzy features, each with 10 bins)

# ============================================================================
# EVALUATION CONFIGURATION
# ============================================================================

EVAL_SAMPLE_SIZE = 500
EVAL_EPISODE_LENGTHS = [300, 600, 1200, 3600]
RANDOM_SEED = 42

AGE_GROUPS = {
    "25-29": (25, 29),
    "30-45": (30, 45),
    "46-60": (46, 60),
    "60-80": (60, 80),
    "80+": (80, 120),
}

# ============================================================================
# PATHS
# ============================================================================

ARTIFACTS_DIR = Path("artifacts")
METRICS_DIR = Path("metrics")
DATA_PATH = Path("data/Patients Data.csv")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
DP_PATH = ARTIFACTS_DIR / "dp_bis_deltabis_agent.npz"

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def get_fuzzy_features(bis_error, deltabis):
    """Build fuzzy features from BIS error and deltaBIS."""
    error_scaled = np.clip(bis_error / 20.0, -1, 1)
    mu_error_neg = max(0, -error_scaled)
    mu_error_zero = max(0, 1.0 - abs(error_scaled))
    mu_error_pos = max(0, error_scaled)

    delta_scaled = np.clip(deltabis / 20.0, -1, 1)
    mu_delta_neg = max(0, -delta_scaled)
    mu_delta_zero = max(0, 1.0 - abs(delta_scaled))
    mu_delta_pos = max(0, delta_scaled)

    return np.array([mu_error_neg, mu_error_zero, mu_error_pos, mu_delta_neg, mu_delta_zero, mu_delta_pos])


def state_to_idx(features):
    """Map fuzzy features to integer state index in [0, NUM_STATES-1]."""
    features = np.asarray(features)
    if features.shape != (6,):
        raise ValueError(f"features must be length-6 vector, got shape {features.shape}")

    bins = np.clip((features * (BINS_PER_FEAT - 1)), 0, BINS_PER_FEAT - 1).astype(int)
    idx = 0
    for b in bins:
        idx = idx * BINS_PER_FEAT + int(b)
    return idx


def get_ce_from_error(bis_error):
    """Inverse PD relation to estimate Ce from BIS error."""
    bis = bis_error + BIS_TARGET
    ratio = (BIS_0 - bis) / BIS_MAX
    ratio = np.clip(ratio, 0.01, 0.99)
    ce = EC50 * (ratio / (1 - ratio)) ** (1 / HILL)
    return ce

# ============================================================================
# DP ALGORITHM: TRAINING SECTION
# ============================================================================

print("=" * 70)
print("DYNAMIC PROGRAMMING: Value Iteration (BIS Error + deltaBIS)")
print("=" * 70)

loaded_V = None
if DP_PATH.exists() and not FORCE_RETRAIN:
    try:
        data = np.load(DP_PATH)
        loaded_V = data["V"]
        print(f"✓ Found existing agent at {DP_PATH}, will warm-start Value Iteration.")
    except Exception as e:
        print(f"Could not load existing agent: {e}")
elif DP_PATH.exists() and FORCE_RETRAIN:
    print(f"↺ FORCE_RETRAIN=True: ignoring existing agent at {DP_PATH} and training from scratch.")

print("\nComputing Transition Matrix (P) and Rewards (R)...")
P = np.zeros((NUM_STATES, len(ACTIONS)), dtype=int)
R = np.zeros((NUM_STATES, len(ACTIONS)))

for s in range(NUM_STATES):
    bins = np.array([
        s // (BINS_PER_FEAT**5),
        (s // (BINS_PER_FEAT**4)) % BINS_PER_FEAT,
        (s // (BINS_PER_FEAT**3)) % BINS_PER_FEAT,
        (s // (BINS_PER_FEAT**2)) % BINS_PER_FEAT,
        (s // BINS_PER_FEAT) % BINS_PER_FEAT,
        s % BINS_PER_FEAT,
    ], dtype=float)
    features = bins / (BINS_PER_FEAT - 1)

    bis_error = (features[2] - features[0]) * 30
    deltabis = (features[5] - features[3]) * 30

    for a_idx, infusion in enumerate(ACTIONS):
        bis_error_next = bis_error - (infusion / max(ACTIONS)) * 15
        deltabis_next = deltabis - (infusion / max(ACTIONS)) * 10

        bis_error_next = np.clip(bis_error_next, -30, 30)
        deltabis_next = np.clip(deltabis_next, -30, 30)

        features_next = get_fuzzy_features(bis_error_next, deltabis_next)
        s_next = state_to_idx(features_next)

        P[s, a_idx] = s_next
        R[s, a_idx] = -(abs(bis_error_next) + 0.5 * abs(deltabis_next))

print("✓ Transition matrix computed")

print("\nExecuting Value Iteration...")
if loaded_V is not None and loaded_V.shape == (NUM_STATES,):
    V = loaded_V.copy()
else:
    V = np.zeros(NUM_STATES)

for ep in range(EPISODES):
    max_delta = 0.0

    for step in range(STEPS_PER_EPISODE):
        V_old = V.copy()
        Q_vals = R + GAMMA * V[P]
        V = np.max(Q_vals, axis=1)

        max_delta = np.max(np.abs(V - V_old))

    if (ep + 1) % 100 == 0:
        print(f"  Episode {ep + 1}/{EPISODES}, step {step + 1}/{STEPS_PER_EPISODE}, max change: {max_delta:.6f}")

print("\nExtracting optimal policy...")
Q_vals = R + GAMMA * V[P]

np.random.seed(RANDOM_SEED)
policy = np.zeros(NUM_STATES, dtype=int)
for s in range(NUM_STATES):
    if np.random.random() < EPSILON:
        policy[s] = np.random.randint(len(ACTIONS))
    else:
        policy[s] = np.argmax(Q_vals[s])

random.seed(RANDOM_SEED)

print(f"\n1. Loading patient data from {DATA_PATH}...")
df = load_data(DATA_PATH)
df = preprocess_data(df)
print(f"   Loaded {len(df)} patients")

print(f"\n2. Sampling {EVAL_SAMPLE_SIZE} patients...")
sample_df = df.sample(n=min(EVAL_SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED)
print(f"   Sampled {len(sample_df)} patients")

print("\n3. Generating Schnider PK/PD parameters...")
eval_df = generate_schnider_dataset(sample_df)
print(f"   Generated parameters for {len(eval_df)} patients")

print(f"\n4. Loading trained DP agent from {DP_PATH}...")
dp_data = np.load(DP_PATH)
policy_eval = dp_data["policy"]
actions_eval = dp_data["actions"]
print(f"   Loaded policy: shape {policy_eval.shape}")

evaluator = DPEvaluator(policy_eval, actions_eval)

print(f"\n5. Evaluating policy at {len(EVAL_EPISODE_LENGTHS)} episode lengths...")
patient_results = []

for idx, (_, patient) in enumerate(eval_df.iterrows()):
    if (idx + 1) % 50 == 0:
        print(f"   Processing patient {idx + 1}/{len(eval_df)}...")

    patient_id = patient["PatientID"]
    age = patient["AgeCategory"]
    age_group = get_age_group(age, AGE_GROUPS)

    results = {}
    for ep_len in EVAL_EPISODE_LENGTHS:
        bis_traj = evaluator.simulate(patient, ep_len)
        metrics = calculate_bis_metrics(bis_traj, BIS_TARGET)
        results[ep_len] = metrics

    patient_results.append(
        {
            "patient_id": patient_id,
            "age": age,
            "age_group": age_group,
            "results": results,
        }
    )

results_df = create_results_dataframe(patient_results, EVAL_EPISODE_LENGTHS)
summary_df = create_summary_by_age_group(results_df, EVAL_EPISODE_LENGTHS, AGE_GROUPS)

print("\n6. Saving evaluation results...")
save_evaluation_results(
    results_df,
    summary_df,
    "dp_bis_deltabis_agent",
    str(METRICS_DIR),
)

print("\n" + "=" * 70)
print("Evaluation complete!")
print(f"  - Results: {len(results_df)} patients")
print(f"  - Age groups: {results_df['AgeGroup'].nunique()}")
print(f"  - Episode lengths: {len(EVAL_EPISODE_LENGTHS)}")
print("=" * 70)

DYNAMIC PROGRAMMING: Value Iteration (BIS Error + deltaBIS)

Computing Transition Matrix (P) and Rewards (R)...
✓ Transition matrix computed

Executing Value Iteration...


# LOAD AND EVALUATE SAVED DP AGENT

In [ ]:
from utils.eval_runner import run_saved_dp_evaluation

results_df = run_saved_dp_evaluation(
    dp_path=DP_PATH,
    evaluator_cls=DPEvaluator,
    load_data_fn=load_data,
    preprocess_data_fn=preprocess_data,
    generate_dataset_fn=generate_schnider_dataset,
    sample_size=100,
)


LOADING SAVED DP AGENT

Loading agent from: artifacts\dp_bis_deltabis_agent.npz
  - Loaded V (value function): shape (1000000,)
  - Loaded policy: shape (1000000,)
  - Loaded P (transitions): shape (1000000, 7)
  - Loaded R (rewards): shape (1000000, 7)
  - Actions available: [0.  0.5 1.  2.  3.  4.  6. ]
  - Gamma (discount): [0.69]

EVALUATING ON POPULATION SAMPLE

Sampling 100 patients from 237630 total...
Generating Schnider parameters for 100 patients...

Running evaluation simulation (120 min per patient)...

EVALUATION RESULTS

Evaluated 100 patients successfully

             MDPE       MDAPE      Wobble  Controlled (%)
count  100.000000  100.000000  100.000000           100.0
mean    89.984640   89.984640    4.050682             0.0
std      0.233030    0.233030    0.127427             0.0
min     89.415586   89.415586    3.791787             0.0
25%     89.831943   89.831943    3.943485             0.0
50%     89.981368   89.981368    4.057370             0.0
75%     90.15267